# Agent playground
Run one agent turn, step through its tool calls, and read the log. Uses the scripted fake model by default.
Set `OPENROUTER_API_KEY` (and `OPENROUTER_MODEL`) in your environment or `backend/.env` to use a real model.
Payments are always the mock provider and training is always the local free backend here: nothing can spend money.
Start Jupyter from `backend/` (Jupyter itself is not a project dependency: `pip install notebook`).

Every turn prints a live log line for each model call, gate decision and tool result, and writes `backend/logs/nb-<time>/t<n>/` (events.jsonl and transcript.md). The conversation is stored in the same tables as the app (source `notebook`).

In [ ]:
import sys, tempfile
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))   # so `app` and `evals` import when started from backend/notebooks
sys.path.insert(0, str(Path.cwd()))          # ...or from backend/

from sqlalchemy import create_engine
from sqlalchemy.orm import sessionmaker
from sqlalchemy.pool import StaticPool

from app.config import settings
from app.db import Base
from app.models import Project
from app.providers.payments.mock import MockPaymentProvider
from app.providers.storage.local import LocalStorage
from app.providers.training.local import LocalBackend
from app.repositories.traces import TraceRepository
from app.services.agent.loop import run_agent
from app.services.agent.tools import ToolContext
from evals.fake import OfflineSource, ReactiveFake
from datetime import datetime, timezone
from app.services.agent.runlog import ConsoleSink, JsonlSink, MarkdownSink, RunLog
from evals.show import LOG_ROOT, show_run, tail


In [ ]:
GOAL = "I want to predict which customers will cancel next month."
STEP_BY_STEP = False   # True: press Enter in the notebook after each model reply

use_real = settings.openrouter_api_key.strip() not in ("", "your-openrouter-key")
if use_real:
    from app.providers.llm.openrouter import OpenRouterClient
    from app.providers.datasets import get_dataset_source
    inner, source = OpenRouterClient(), get_dataset_source()   # model id comes from OPENROUTER_MODEL
    print("Using OpenRouter model", settings.openrouter_model)
else:
    inner, source = ReactiveFake({"fake": {"behavior": "dataset", "query": "customer churn", "target": "left"}}), OfflineSource()
    print("Using the scripted fake model (no key set)")


class StepByStep:
    """Optional: pause after each model reply (set STEP_BY_STEP = True above)."""
    def __init__(self, inner):
        self.inner = inner
        self.model = getattr(inner, "model", type(inner).__name__)

    def chat(self, messages, tools):
        r = self.inner.chat(messages, tools)
        if STEP_BY_STEP:
            input("Enter to continue...")
        return r


engine = create_engine("sqlite://", poolclass=StaticPool, connect_args={"check_same_thread": False})
Base.metadata.create_all(engine)
session = sessionmaker(engine, expire_on_commit=False)()
project = Project(goal=GOAL)
session.add(project)
session.commit()
tmp = tempfile.mkdtemp()
llm = StepByStep(inner)
STAMP = datetime.now(timezone.utc).strftime("%Y%m%d-%H%M%S")
turn_no = 0


def turn(message=None, approve=()):
    global turn_no
    turn_no += 1
    folder = LOG_ROOT / f"nb-{STAMP}" / f"t{turn_no}"
    log = RunLog([ConsoleSink(sys.stdout), JsonlSink(folder), MarkdownSink(folder)], log_id=f"nb-{STAMP}/t{turn_no}")  # add log_prompts=True to keep full prompts
    ctx = ToolContext(session, project, LocalStorage(tmp), MockPaymentProvider(), "mock", LocalBackend(), source, set(approve))
    r = run_agent(ctx, llm, message, log=log, source="notebook", meta={"notebook": "agent_playground"})
    log.run_end(project.status)
    log.close()
    print("RESULT:", r.status, r.text or r.card)
    print("Project status:", project.status, "| log:", folder)
    return r


## One turn
Send the goal. If the agent needs approval, the result is `awaiting_approval` and shows the card.

In [ ]:
r = turn(GOAL)

## Approve choosing the dataset
The only approval this playground offers. Run it only if the result above was an approval card for `select_dataset`.

In [ ]:
r = turn(None, approve={"select_dataset"})

## Keep going
The agent continues from the saved state. The quote is never approved here, so nothing paid can happen.

In [ ]:
r = turn("Yes, please go ahead with the next step.")

## Trace and past runs
The audit trace for this project (every step with its run id). `show_run('nb-<time>/t1', tools=True)` shows one turn's tool calls and results; `tail()` shows the newest log lines.

In [ ]:
for st in TraceRepository(session).steps(project.id):
    print(f"{st.seq:>3} {st.kind:<12} run={str(st.run_id)[:6]}  {str(st.content)[:150]}")

In [ ]:
print(tail(15))
# show_run(f'nb-{STAMP}/t1', tools=True, display=True)   # tool calls and results of the first turn
# show_run(f'nb-{STAMP}/t1', md=True, display=True)      # readable transcript